Análisis exploratorio de datos – ENFR 2018
Trabajo práctico de Fundamentos de Ciencia de Datos.

## 1. Carga y descripción del dataset

In [ ]:
# Carga del conjunto de datos
import os
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Descomprimimos el archivo si todavía no existe la carpeta
if not os.path.exists("ENFR2018"):
    with zipfile.ZipFile("ENFR2018_baseusuario.zip", "r") as zip_ref:
        zip_ref.extractall("ENFR2018")
    print("Archivo descomprimido correctamente.")
else:
    print("La carpeta ENFR2018 ya existe.")


df = pd.read_csv("ENFR2018_baseusuario.zip", sep="|")


# Revisamos las dimensiones y los tipos de datos
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])


df.info()

# Nombres de las columnas
print(df.columns.tolist())



## 2. Revisión inicial de valores faltantes

In [ ]:
#Análisis de valores nulos
nulos = df.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

porcentaje_nulos = (df.isnull().sum() / len(df) * 100)
print(porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False))

porcentaje_nulos = df.isnull().sum() / len(df) * 100

print("Sin nulos:", (porcentaje_nulos == 0).sum())
print("Hasta 10%:", ((porcentaje_nulos > 0) & (porcentaje_nulos <= 10)).sum())
print("Entre 10% y 50%:", ((porcentaje_nulos > 10) & (porcentaje_nulos <= 50)).sum())
print("Entre 50% y 90%:", ((porcentaje_nulos > 50) & (porcentaje_nulos <= 90)).sum())
print("Más del 90%:", (porcentaje_nulos > 90).sum())

## 3. Análisis exploratorio por dimensiones

### 3.1. Estadísticos descriptivos de las dimensiones

In [ ]:
# Estadísticos descriptivos de las dimensiones
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


# Resumen de la variable edad
promedio_edad = df['bhch04'].mean()
desviacion_edad = df['bhch04'].std()
edad_maxima = df['bhch04'].max()

print(
    f'Promedio [X]: {promedio_edad:.2f} | Desviación estándar [Y]: {desviacion_edad:.2f} | Máximo [Z]: {edad_maxima}'
)

# Edad (bhch04)
edad = df['bhch04'].dropna()

media_edad = edad.mean()
mediana_edad = edad.median()

print(f"Edad - Media: {media_edad:.2f} años | Mediana: {mediana_edad:.1f} años")

# Histograma de edad con media y mediana
plt.figure(figsize=(10, 5))

sns.histplot(
    edad,
    bins=30,
    kde=True,
    color='skyblue',
    edgecolor='black'
)

plt.axvline(
    media_edad,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f'Media: {media_edad:.2f} años',
)

plt.axvline(
    mediana_edad,
    color='green',
    linestyle='-',
    linewidth=2,
    label=f'Mediana: {mediana_edad:.1f} años',
)

plt.title(
    'Fig. 1. Distribución de la variable EDAD en la población encuestada (ENFR 2018)',
    fontsize=13,
    fontweight='bold',
    pad=12,
)

plt.xlabel('Edad (años)', fontsize=12)
plt.ylabel('Frecuencia (Cantidad de personas)', fontsize=12)
plt.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig('fig1_distribucion_edad.png', dpi=300)
plt.show()


# Sexo (bhch03: 1=Varón, 2=Mujer)
sexo_counts = df['bhch03'].value_counts()
sexo_pct = df['bhch03'].value_counts(normalize=True) * 100

print("\nDistribución por Sexo:")
print(pd.DataFrame({
    'Cantidad': sexo_counts,
    'Porcentaje (%)': sexo_pct.round(2)
}))


# Nivel Educativo Agrupado
edu_pct = df['nivel_instruccion_agrupado'].value_counts(normalize=True) * 100

print("\nNivel Educativo Agrupado (%):")
print(edu_pct.round(2))


# Ingresos del Hogar (bhih01)
print(
    f"\nIngreso del Hogar (bhih01) - Media: ${df['bhih01'].mean():.2f} "
    f"| Mediana: ${df['bhih01'].median():.2f}"
)


# Dimensión 2: Estilo de vida y hábitos
# ---------------------------------------------------------

print("\n=== 2. DIMENSIÓN HÁBITOS ===")

# Tabaco
fuma_pct = df['consumo_tabaco_100'].value_counts(normalize=True) * 100

print("Condición de Fumador (%):")
print(fuma_pct.round(2))


# Consumo de Alcohol en últimos 30 días
alcohol_30d = (df['bica02'] == 1).mean() * 100

print(f"\nPorcentaje que consumió alcohol en los últimos 30 días: {alcohol_30d:.2f}%")


# Agregado de Sal en la mesa
sal_pct = df['bial02'].value_counts(normalize=True) * 100

print("\nUso de sal en la mesa (%):")
print(sal_pct.round(2))


# Consumo diario de Frutas y Verduras
media_fv = df['promedio_fv_diario'].mean()
mediana_fv = df['promedio_fv_diario'].median()
cumple_5_porciones = (df['consumo_fv'] == 1).mean() * 100

print(
    f"\nConsumo de Frutas/Verduras - Porciones diarias "
    f"(Media): {media_fv:.2f} | Mediana: {mediana_fv:.2f}"
)

print(f"Porcentaje que consume al menos 5 porciones diarias: {cumple_5_porciones:.2f}%")


# Dimensión 3: Salud y enfermedades no transmisibles
# ---------------------------------------------------------

print("\n=== 3. DIMENSIÓN SALUD ===")

# Prevalencia de Hipertensión
hip_pct = (df['prevalencia_hipertension'] == 1).mean() * 100

print(f"Prevalencia de Hipertensión (Autorreporte): {hip_pct:.2f}%")


# Prevalencia de Diabetes
diab_pct = (df['prevalencia_diabetes'] == 1).mean() * 100

print(f"Prevalencia de Diabetes (Autorreporte): {diab_pct:.2f}%")


# Prevalencia de Colesterol Alto
col_pct = (df['prevalencia_colesterol'] == 1).mean() * 100

print(f"Prevalencia de Colesterol Alto (Autorreporte): {col_pct:.2f}%")


# Índice de Masa Corporal (IMC)
imc_media = df['imc'].mean()
imc_mediana = df['imc'].median()
imc_cat_pct = df['imc_categorias'].value_counts(normalize=True) * 100

print(
    f"\nIMC continuo - Media: {imc_media:.2f} kg/m² "
    f"| Mediana: {imc_mediana:.2f} kg/m²"
)

print("Categorías de IMC (%):")
print(imc_cat_pct.round(2))


# Revisión de la calidad de los datos
# ---------------------------------------------------------

print("\n=== 4. CALIDAD DE DATOS (NULOS Y OUTLIERS) ===")

# Variables seleccionadas para la revisión de nulos
vars_analisis = [
    'bhch04', 'bhch03', 'nivel_instruccion_agrupado', 'bhih01',
    'consumo_tabaco_100', 'bica02', 'bial02', 'promedio_fv_diario',
    'prevalencia_hipertension', 'prevalencia_diabetes',
    'prevalencia_colesterol', 'imc'
]

# Conteos de valores nulos
nulos_df = pd.DataFrame({
    'Valores Nulos': df[vars_analisis].isnull().sum(),
    'Porcentaje (%)': (df[vars_analisis].isnull().mean() * 100).round(2)
})

print("Reporte de Valores Nulos por Variable:")
print(nulos_df)


# Rangos Mínimos y Máximos
print("\nRangos observados (Mínimo - Máximo):")

for col_cont in ['bhch04', 'bhih01', 'promedio_fv_diario', 'imc']:
    if col_cont in df.columns:
        print(f"{col_cont}: Min = {df[col_cont].min()} | Max = {df[col_cont].max()}")

### 3.2. Dimensión 1: análisis gráfico de ingresos del hogar

In [ ]:
# Distribución de los ingresos del hogar (bhih01)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")


# Histograma de ingresos
plt.figure(figsize=(10, 5))

# Limitamos el eje para que los valores extremos no compriman el gráfico
limite_x = df['bhih01'].quantile(0.98)

# Histograma con curva de densidad
sns.histplot(
    df['bhih01'],
    bins=40,
    binrange=(0, limite_x),
    color='#1f77b4',
    kde=True,
)

# Media y mediana
media_val = df['bhih01'].mean()
mediana_val = df['bhih01'].median()

# Agregamos las líneas de referencia
plt.axvline(
    mediana_val,
    color='green',
    linestyle='--',
    linewidth=2,
    label=f'Mediana: ${mediana_val:,.0f}',
)

plt.axvline(
    media_val,
    color='red',
    linestyle=':',
    linewidth=2,
    label=f'Media: ${media_val:,.0f}',
)

plt.title(
    'Distribución del Ingreso Total del Hogar (bhih01)',
    fontsize=13,
    fontweight='bold',
    pad=12,
)

plt.xlim(0, limite_x)
plt.xlabel('Ingreso Total del Hogar ($)', fontsize=11)
plt.ylabel('Cantidad de Hogares', fontsize=11)
plt.legend(fontsize=11)

plt.tight_layout()
plt.savefig('bhih01_histograma_final.png', dpi=300)
plt.show()



# Distribución por tramos de ingreso

plt.figure(figsize=(9, 5))

bins_fijos = [-np.inf, 10000, 20000, 35000, 60000, np.inf]

labels_fijos = [
    'Hasta $10.000',
    '$10.001 a $20.000',
    '$20.001 a $35.000',
    '$35.001 a $60.000',
    'Más de $60.000',
]

df['rango_ingreso_completo'] = pd.cut(
    df['bhih01'], bins=bins_fijos, labels=labels_fijos
)

pct_tramos = (
    df['rango_ingreso_completo'].value_counts(normalize=True).sort_index() * 100
)

bars = plt.bar(
    pct_tramos.index,
    pct_tramos.values,
    color='#3182bd',
    edgecolor='black',
    width=0.55,
)

plt.title(
    'Distribución de Hogares por Tramos de Ingreso',
    fontsize=13,
    fontweight='bold',
    pad=15,
)

plt.ylabel('Porcentaje de Hogares (%)', fontsize=11)
plt.xlabel('Tramo de Ingreso Monetario del Hogar', fontsize=11)
plt.ylim(0, max(pct_tramos.values) + 8)

for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        yval + 1,
        f'{yval:.1f}%',
        ha='center',
        va='bottom',
        fontweight='bold',
    )

plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('rango_ingreso_barras_final.png', dpi=300)
plt.show()

### 3.3. Dimensión 1: comparación de métodos de imputación

In [ ]:
# Comparación de métodos de imputación para bhih01

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Identificamos los ingresos imputados oficialmente por INDEC
mascara_imputados = pd.to_numeric(
    df['imputado'], errors='coerce'
).eq(1)

# Creamos una copia y enmascaramos los valores imputados
df['bhih01_clean'] = pd.to_numeric(
    df['bhih01'], errors='coerce'
).copy()

df.loc[mascara_imputados, 'bhih01_clean'] = np.nan

# Conservamos los valores oficiales como referencia
referencia_oficial = df.loc[mascara_imputados, 'bhih01']

# Imputación por media y mediana
media_val = df['bhih01_clean'].mean()
mediana_val = df['bhih01_clean'].median()

df['bhih01_media'] = df['bhih01_clean'].fillna(media_val)
df['bhih01_mediana'] = df['bhih01_clean'].fillna(mediana_val)

# Preparación de variables auxiliares para KNN
datos_knn = pd.DataFrame({
    'edad': pd.to_numeric(df['bhch04'], errors='coerce'),
    'sexo': df['bhch03'].map({1: 0, 2: 1}),
    'ingreso': df['bhih01_clean']
})

# Estandarizamos las variables para que tengan escalas comparables
escalador = StandardScaler()
datos_escalados = escalador.fit_transform(datos_knn)

# Aplicamos KNN
imputer = KNNImputer(n_neighbors=5)
datos_imputados = imputer.fit_transform(datos_escalados)

# Recuperamos la escala original del ingreso
df['bhih01_knn'] = (
    datos_imputados[:, 2] * escalador.scale_[2]
    + escalador.mean_[2]
)

# Si KNN no pudo imputar algún registro, usamos la mediana
df['bhih01_knn'] = df['bhih01_knn'].fillna(mediana_val)

# Comparación de estadísticas
tabla_comparativa = pd.DataFrame({
    'Original (INDEC)': df['bhih01'].describe(),
    'Mediana': df['bhih01_mediana'].describe(),
    'Media': df['bhih01_media'].describe(),
    'KNN': df['bhih01_knn'].describe()
})

print("COMPARACIÓN DE MÉTODOS")
display(tabla_comparativa.round(2))

# Comparación con los valores imputados oficialmente
comparacion = pd.DataFrame({
    'Mediana': df.loc[mascara_imputados, 'bhih01_mediana'],
    'Media': df.loc[mascara_imputados, 'bhih01_media'],
    'KNN': df.loc[mascara_imputados, 'bhih01_knn'],
    'Referencia INDEC': referencia_oficial
})

mae = (
    comparacion[['Mediana', 'Media', 'KNN']]
    .sub(comparacion['Referencia INDEC'], axis=0)
    .abs()
    .mean()
)

print("\nError absoluto medio respecto de la imputación oficial:")
print(mae.round(2))

In [ ]:
#ANALISIS DE IMPUTACIÓN DE DATOS PARA LA VARIABLE bhih01
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

# 1. Limpieza de datos no válidos / códigos NsNc a NaN real
df['bhih01_clean'] = df['bhih01'].replace([99, 999, 9999, 999999, -1], np.nan)

# 2. Generación de las 3 imputaciones
mediana_val = df['bhih01_clean'].median()
media_val = df['bhih01_clean'].mean()

df['bhih01_mediana'] = df['bhih01_clean'].fillna(mediana_val)
df['bhih01_media'] = df['bhih01_clean'].fillna(media_val)

imputer_knn = KNNImputer(n_neighbors=5)
cols_aux = ['bhch04', 'bhch03', 'bhih01_clean']
df['bhih01_knn'] = imputer_knn.fit_transform(df[cols_aux])[:, 2]

# 3. Consolidación de estadísticos descriptivos en un solo DataFrame
tabla_comparativa = pd.DataFrame({
    'Original (sin nulos)': df['bhih01_clean'].describe(),
    'Imp. Mediana': df['bhih01_mediana'].describe(),
    'Imp. Media': df['bhih01_media'].describe(),
    'Imp. KNN (k=5)': df['bhih01_knn'].describe(),
})

# Formato de visualización claro
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("=== TABLA COMPARATIVA DE ESTADÍSTICOS DESCRIPTIVOS (bhih01) ===")
print(tabla_comparativa)

#APROBADO: ANALISIS DE IMPUTACIÓN DE DATOS PARA LA VARIABLE bhih01
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer

sns.set_theme(style="whitegrid")

# 1. Convertir códigos de "No responde" / "Sin dato" a NaN reales
# (Ajustar los códigos si en tu dataset tienen otra convención)
df['bhih01_clean'] = df['bhih01'].replace([99, 999, 9999, 999999, -1], np.nan)

# 2. Calcular e imputar sobre la variable limpia
mediana_val = df['bhih01_clean'].median()
media_val = df['bhih01_clean'].mean()

df['bhih01_mediana'] = df['bhih01_clean'].fillna(mediana_val)
df['bhih01_media'] = df['bhih01_clean'].fillna(media_val)

imputer_knn = KNNImputer(n_neighbors=5)
cols_aux = ['bhch04', 'bhch03', 'bhih01_clean']
df['bhih01_knn'] = imputer_knn.fit_transform(df[cols_aux])[:, 2]

# 3. Definir límite superior razonable (percentil 98) para ignorar outliers extremos en la visualización
limite_x = df['bhih01_clean'].quantile(0.98)

# 4. Generar histogramas con rango y cantidad de bins definidos
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

# Original
sns.histplot(
    df['bhih01_clean'].dropna(),
    ax=axes[0, 0],
    bins=30,
    binrange=(0, limite_x),
    color='gray',
)
axes[0, 0].set_title(
    '1. Original (Sin Nulos - Omite 43% de datos)', fontweight='bold'
)

# Mediana
sns.histplot(
    df['bhih01_mediana'],
    ax=axes[0, 1],
    bins=30,
    binrange=(0, limite_x),
    color='#2ca02c',
)
axes[0, 1].set_title(
    f'2. Imputado por Mediana (${mediana_val:,.0f})', fontweight='bold'
)

# Media
sns.histplot(
    df['bhih01_media'],
    ax=axes[1, 0],
    bins=30,
    binrange=(0, limite_x),
    color='#d62728',
)
axes[1, 0].set_title(
    f'3. Imputado por Media (${media_val:,.0f})', fontweight='bold'
)

# KNN
sns.histplot(
    df['bhih01_knn'],
    ax=axes[1, 1],
    bins=30,
    binrange=(0, limite_x),
    color='#1f77b4',
)
axes[1, 1].set_title('4. Imputado por KNN (k=5)', fontweight='bold')

for ax in axes.flat:
  ax.set_xlabel('Ingreso del Hogar ($)', fontsize=11)
  ax.set_ylabel('Frecuencia (Hogares)', fontsize=11)

plt.suptitle(
    'Comparación Real de Histogramas de Ingresos (bhih01) Post-Imputación',
    fontsize=14,
    fontweight='bold',
    y=0.98,
)
plt.tight_layout()
plt.show()

### 3.4. Dimensión 2: estilos de vida y habitos

In [ ]:
# Condición de fumador
no_fumador = (df['consumo_tabaco_100'] == 3).sum()
fumador_actual = (df['consumo_tabaco_100'] == 1).sum()
ex_fumador = (df['consumo_tabaco_100'] == 2).sum()

# Nivel de actividad física
af_bajo = (df['nivel_actividad_fisica'] == 3).sum()
af_medio_alto = df['nivel_actividad_fisica'].isin([1, 2]).sum()


# 2. GENERACIÓN DE LOS GRÁFICOS DE TORTA DIMENSIÓN 2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico 1: Condición de Fumador
labels_tabaco = ['No fumador', 'Fumador actual', 'Ex fumador']
sizes_tabaco = [no_fumador, fumador_actual, ex_fumador]
colors_tabaco = ['#4c72b0', '#dd8452', '#55a868']

axes[0].pie(
    sizes_tabaco,
    labels=labels_tabaco,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors_tabaco,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1},
)
axes[0].set_title('Condicion de Fumador', fontsize=11)

# Gráfico 2: Nivel de Actividad Física
labels_af = ['Nivel bajo', 'Nivel medio / alto']
sizes_af = [af_bajo, af_medio_alto]
colors_af = ['#4c72b0', '#dd8452']

axes[1].pie(
    sizes_af,
    labels=labels_af,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors_af,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1},
)
axes[1].set_title('Nivel de Actividad Fisica', fontsize=11)

plt.tight_layout()
plt.show()

### 3.5. Dimensión 3: salud y enfermedades no transmisibles

In [ ]:
#Dimensión 3 graficos de torta y diagnostico de nulos
#Dimensión 3: Salud y Enfermedades No Transmisibles (ECNT): hipertensión, diabetes, IMC/obesidad y colesterol
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# 1. CÁLCULO DE METRICAS (SALUD Y ESTADO NUTRICIONAL)
vars_parrafo = [
    'imc_categorias',
    'prevalencia_hipertension',
    'prevalencia_diabetes',
    'prevalencia_colesterol',
]

print("=== DIAGNÓSTICO DE NULOS EN VARIABLES DEL PÁRRAFO ===")
for var in vars_parrafo:
  if var in df.columns:
    nan_count = df[var].isnull().sum()
    nsnc_count = df[var].isin([99, 999]).sum()
    total_faltantes = nan_count + nsnc_count
    pct = (total_faltantes / len(df)) * 100

    print(f'Variable: {var}')
    print(f'  • Outliers: No aplican (Variable Categórica)')
    print(f'  • Registros Faltantes (NaN + Ns/Nc): {total_faltantes}')
    print(f'  • Porcentaje de Nulos: {pct:.2f}%\n')
    

# Estado Nutricional (imc_categorias: 1=Bajo/Normopeso, 2=Sobrepeso, 3=Obesidad)
x_normopeso = (df['imc_categorias'] == 1).mean() * 100
y_sobrepeso = (df['imc_categorias'] == 2).mean() * 100
z_obesidad = (df['imc_categorias'] == 3).mean() * 100

# Diagnósticos Médicos / ECNT (1 = Sí)
a_hipertension = (df['prevalencia_hipertension'] == 1).mean() * 100
b_diabetes = (df['prevalencia_diabetes'] == 1).mean() * 100
c_colesterol = (df['prevalencia_colesterol'] == 1).mean() * 100


print("=" * 45)
print("  VALORES SALUD Y ESTADO NUTRICIONAL (ECNT)")
print("=" * 45)
print(f"[X] Normopeso / Peso Bajo : {x_normopeso:.2f}%")
print(f"[Y] Sobrepeso             : {y_sobrepeso:.2f}%")
print(f"[Z] Obesidad              : {z_obesidad:.2f}%")
print(f"[A] Hipertensión          : {a_hipertension:.2f}%")
print(f"[B] Diabetes / Glucemia   : {b_diabetes:.2f}%")
print(f"[C] Colesterol Elevado    : {c_colesterol:.2f}%")
print("=" * 45)

# =========================================================
# 2. GRAFICOS DE TORTA (VARIABLES CON POCAS CATEGORIAS)
# =========================================================
sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---------------------------------------------------------
# Gráfico 1: Torta - Nivel de Actividad Física
# (nivel_actividad_fisica: 1=Alto, 2=Medio, 3=Bajo)
# ---------------------------------------------------------
af_counts = df['nivel_actividad_fisica'].value_counts().sort_index()
af_labels = ['Alto', 'Medio', 'Bajo']
af_colors = ['#2ca02c', '#1f77b4', '#d62728']

axes[0].pie(
    af_counts,
    labels=af_labels,
    autopct='%1.1f%%',
    startangle=140,
    colors=af_colors,
    explode=(0, 0, 0.05),
    shadow=True,
    textprops={'fontsize': 11, 'weight': 'bold'},
)
axes[0].set_title(
    'Nivel de Actividad Física\n(nivel_actividad_fisica)',
    fontsize=13,
    fontweight='bold',
    pad=15,
)

# ---------------------------------------------------------
# Gráfico 2: Torta - Categorías de Estado Nutricional (IMC)
# ---------------------------------------------------------
imc_values = [x_normopeso, y_sobrepeso, z_obesidad]
imc_labels = ['Normopeso / Bajo', 'Sobrepeso', 'Obesidad']
imc_colors = ['#2ca02c', '#ff7f0e', '#d62728']

axes[1].pie(
    imc_values,
    labels=imc_labels,
    autopct='%1.1f%%',
    startangle=140,
    colors=imc_colors,
    explode=(0, 0.03, 0.05),
    shadow=True,
    textprops={'fontsize': 11, 'weight': 'bold'},
)
axes[1].set_title(
    'Categorías de Estado Nutricional\n(imc_categorias)',
    fontsize=13,
    fontweight='bold',
    pad=15,
)

plt.tight_layout()
plt.show()

In [ ]:
## grafico enfermedades no transmisibles (ECNT)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Cálculo de prevalencias por autorreporte (1 = Sí)
prev_hip = (df['prevalencia_hipertension'] == 1).mean() * 100
prev_col = (df['prevalencia_colesterol'] == 1).mean() * 100
prev_diab = (df['prevalencia_diabetes'] == 1).mean() * 100

# 2. Datos y etiquetas para el gráfico
ecnt_labels = [
    'Hipertensión\nArterial',
    'Colesterol\nElevado',
    'Diabetes /\nGlucemia Alt.',
]
ecnt_values = [prev_hip, prev_col, prev_diab]
ecnt_colors = ['#d62728', '#ff7f0e', '#1f77b4']

# 3. Generación del gráfico de barras
plt.figure(figsize=(7, 5))
bars = plt.bar(
    ecnt_labels,
    ecnt_values,
    color=ecnt_colors,
    width=0.45,
    edgecolor='black',
    linewidth=0.8,
)

# Etiquetas porcentuales sobre las barras
for bar in bars:
  yval = bar.get_height()
  plt.text(
      bar.get_x() + bar.get_width() / 2.0,
      yval + 0.8,
      f'{yval:.2f}%',
      ha='center',
      va='bottom',
      fontsize=10,
      fontweight='bold',
  )

plt.title(
    'Prevalencia de Enfermedades No Transmisibles (Autorreporte)',
    fontsize=12,
    fontweight='bold',
    pad=15,
)
plt.ylabel('Porcentaje de la Población (%)', fontsize=11)
plt.ylim(0, max(ecnt_values) + 8)

plt.tight_layout()
plt.show()

## 4. Planteo y análisis de hipótesis

### 4.1. Hipótesis 1: ingresos del hogar (univariada)

In [ ]:
# Hipótesis 1: ingresos del hogar
#ingresos en funcion de si es pobre o no "bhih01"
sueldo_min = 10000
sueldos = df["bhih01"]
sueldos_count = [
    (sueldos <= sueldo_min).sum(),
    (sueldos > sueldo_min).sum(),
]
labels = ["Hasta 10.000", "Mayor a 10.000"]

# Crear el gráfico de torta
plt.pie(sueldos_count, labels=labels, autopct='%1.1f%%', startangle=90)

# Agregar un título al gráfico
plt.title("Proporción de ingresos")

# Mostrar el gráfico
plt.show()

### 4.2. Hipótesis 2: actividad física y consumo de tabaco (bivariada)

In [ ]:
# Hipótesis 2: actividad física según condición de fumador
# Actividad física según condición de fumador
actividad_labels = {1: 'Alto', 2: 'Medio', 3: 'Bajo'}
fumador_labels = {1: 'Fumador actual', 2: 'Ex-fumador', 3: 'No fumador'}

# Se consideran solo las respuestas válidas
datos_actividad = df[
    df['nivel_actividad_fisica'].isin(actividad_labels.keys()) &
    df['consumo_tabaco_100'].isin(fumador_labels.keys())
].copy()

tabla_frecuencias = pd.crosstab(
    datos_actividad['consumo_tabaco_100'],
    datos_actividad['nivel_actividad_fisica']
).reindex(
    index=fumador_labels.keys(),
    columns=actividad_labels.keys(),
    fill_value=0
)

categorias_fumador = tabla_frecuencias.index.tolist()
fig, axes = plt.subplots(
    1,
    len(categorias_fumador),
    figsize=(6 * len(categorias_fumador), 6),
)
axes = np.atleast_1d(axes)
colores_actividad = ['#2ca02c', '#1f77b4', '#d62728']

for ax, categoria_fumador in zip(axes, categorias_fumador):
    ax.pie(
        tabla_frecuencias.loc[categoria_fumador],
        labels=list(actividad_labels.values()),
        autopct='%1.1f%%',
        startangle=140,
        colors=colores_actividad,
        textprops={'fontsize': 10, 'weight': 'bold'},
    )
    etiqueta_fumador = fumador_labels.get(
        categoria_fumador,
        f'Código {categoria_fumador}',
    )
    ax.set_title(etiqueta_fumador)

fig.suptitle('Nivel de actividad física por condición de fumador', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3. Hipótesis 3: hábitos y estado de salud (multivariada)

In [ ]:
# Hipótesis 3: hábitos y autopercepción del estado de salud
# El estado de salud de un individuo se puede ver asociado a su nivel de actividad física,
# tipo de alimentación y frecuencia tomando alcohol
# Usamos PCA para visualizar las relaciones entre variables

# Seleccionamos las variables de la hipótesis 3
columnas = ['nivel_actividad_fisica', 'bial08', 'bica03_01', 'bisg01']

# Excluimos los registros con código 99 en bial08 y eliminamos nulos
Hipo = df.loc[df['bial08'] != 99, columnas].dropna().copy()

# Renombramos las variables
Hipo.rename(columns={
    'nivel_actividad_fisica': 'Actividad Física',
    'bial08': 'Dieta',
    'bica03_01': 'Frecuencia Alcohol',
    'bisg01': 'Estado de Salud'
}, inplace=True)


# Estandarizamos los datos
scaler = StandardScaler()
datos_escalados = scaler.fit_transform(Hipo)

# Aplicamos PCA
pca = PCA(n_components=2)
componentes = pca.fit_transform(datos_escalados)

# Generamos el gráfico PCA con las proyecciones de las variables
plt.figure(figsize=(10, 8))

# Puntos de las observaciones
plt.scatter(
    componentes[:, 0],
    componentes[:, 1],
    alpha=0.4,
    color='steelblue'
)

# Proyección de las variables originales
loadings = pca.components_.T

for i, variable in enumerate(Hipo.columns):
    plt.arrow(
        0, 0,
        loadings[i, 0] * 3,
        loadings[i, 1] * 3,
        color='red',
        width=0.008,
        head_width=0.08,
        length_includes_head=True
    )

    plt.text(
        loadings[i, 0] * 3.2,
        loadings[i, 1] * 3.2,
        variable,
        color='red',
        fontsize=10,
        fontweight='bold'
    )

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA: Hábitos y autopercepción del estado de salud')
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.grid(alpha=0.3)
plt.show()

print("Dimensiones de Hipo:", Hipo.shape)

print("\nEstadísticas de las variables:")
display(Hipo.describe())

print("\nVarianza explicada:")
print(pca.explained_variance_ratio_)

print("\nMáximos y mínimos de los componentes:")
print(pd.DataFrame(componentes, columns=['PC1', 'PC2']).describe())

print("\nProyección de las variables:")
display(pd.DataFrame(
    pca.components_.T,
    index=Hipo.columns,
    columns=['PC1', 'PC2']
))

### 4.4. Hipótesis 4: consumo de frutas y verduras (univariada)

In [ ]:
#Hipotesis 4:
#Consumo de frutas menor a 5 porciones diarias
fv_min = 5
fv = df["promedio_fv_diario"]
frutas_count = [
    (fv < fv_min).sum(),
    (fv >= fv_min).sum(),
]
labels = ["Menos de 5", "5 o más"]

# Crear el gráfico de torta
plt.pie(frutas_count, labels=labels, autopct='%1.1f%%', startangle=90)

# Agregar un título al gráfico
plt.title("Proporción de consumo de frutas y verduras")

# Mostrar el gráfico
plt.show()

# Cantidad de personas según porciones diarias promedio
porciones = df['promedio_fv_diario'].dropna()
porciones_bins = pd.cut(
    porciones,
    bins=[-np.inf, 1, 2, 3, 4, 5, np.inf],
    labels=['Menos de 1', '1 a <2', '2 a <3', '3 a <4', '4 a <5', '5 o más'],
    right=False,
)
conteo_porciones = porciones_bins.value_counts(sort=False)

fig, ax = plt.subplots(figsize=(9, 5))
barras_porciones = ax.bar(
    conteo_porciones.index.astype(str),
    conteo_porciones.values,
    color='#2ca02c',
)
ax.bar_label(barras_porciones, padding=3)
ax.set_title('Cantidad de personas según porciones diarias de frutas y verduras')
ax.set_xlabel('Porciones diarias promedio')
ax.set_ylabel('Cantidad de personas')
plt.tight_layout()
plt.show()

### 4.5. Hipótesis 5: IMC y actividad física (bivariada)

In [ ]:
# Hipótesis 5: IMC según nivel de actividad física
# Comparamos las categorías de IMC según el nivel de actividad física
labels_imc = ["Bajo", "Normal", "Alto"]
categorias_imc = [1, 2, 3]
actividad_labels = {1: "Alto", 2: "Medio", 3: "Bajo"}

fig, axes = plt.subplots(1, len(actividad_labels), figsize=(18, 6))

for ax, (nivel_actividad, etiqueta_actividad) in zip(axes, actividad_labels.items()):
    imc_actividad = df.loc[
        df["nivel_actividad_fisica"] == nivel_actividad,
        "imc_categorias",
    ]
    indice_count = [
        (imc_actividad == categoria).sum()
        for categoria in categorias_imc
    ]


    ax.pie(
        indice_count,
        labels=labels_imc,
        autopct="%1.1f%%",
        startangle=90,
    )
    ax.set_title(f"Actividad física: {etiqueta_actividad}")

fig.suptitle("Proporción de índice de masa corporal por nivel de actividad física")
plt.tight_layout()
plt.show()


### 4.6. Hipótesis 6: relación entre variables de salud (PCA)

In [ ]:
# Hipótesis 6: relación entre variables de salud
# Existe una relación lineal entre el índice de masa corporal, la presión sistólica,
# el perímetro de la cintura, la edad y los valores de glucemia capilar registrada en ayunas

# Aplicamos PCA a las variables seleccionadas

columnas = [
    'imc',
    'bima04_01_a',
    'bima13',
    'bhch04',
    'bimq05'
]

# Seleccionamos las variables y quitamos registros incompletos
Hipo = df[columnas].dropna().copy()

Hipo.rename(columns={
    'imc': 'Indice de Masa Corporal',
    'bima04_01_a': 'Presion Sistolica',
    'bima13': 'Perimetro de Cintura',
    'bhch04': 'Edad',
    'bimq05': 'Glucemia en Ayunas'
}, inplace=True)

# Estandarizamos las variables
scaler = StandardScaler()
datos_escalados = scaler.fit_transform(Hipo)

# Aplicamos PCA
pca = PCA(n_components=2)
componentes = pca.fit_transform(datos_escalados)

# Generamos el gráfico
plt.figure(figsize=(10, 8))

# Representamos las observaciones
plt.scatter(
    componentes[:, 0],
    componentes[:, 1],
    alpha=0.4,
    color='steelblue'
)

# Proyección de las variables originales
loadings = pca.components_.T

for i, variable in enumerate(Hipo.columns):
    plt.arrow(
        0, 0,
        loadings[i, 0] * 3,
        loadings[i, 1] * 3,
        color='red',
        width=0.008,
        head_width=0.08,
        length_includes_head=True
    )

    plt.text(
        loadings[i, 0] * 3.2,
        loadings[i, 1] * 3.2,
        variable,
        color='red',
        fontsize=10,
        fontweight='bold'
    )

plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('PCA: Relación entre variables de salud')

plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)

plt.grid(alpha=0.3)
plt.show()

#### 4.6.1. Análisis complementario: IMC, cintura y glucemia

In [ ]:
# Hipótesis 6: análisis complementario (IMC, cintura y glucemia)
# Repetimos el PCA con estas tres variables

columnas = [
    'imc',
    'bima13',
    'bimq05'
]

Hipo = df.loc[:, columnas].dropna().copy()

Hipo.rename(columns={
    'imc': 'Indice de Masa Corporal',
    'bima13': 'Perimetro de Cintura',
    'bimq05': 'Glucemia en Ayunas'
}, inplace=True)

# Estandarizamos las variables
scaler = StandardScaler()
datos_escalados = scaler.fit_transform(Hipo)

# Aplicamos PCA
pca = PCA(n_components=2)
componentes = pca.fit_transform(datos_escalados)

# Generamos el gráfico
plt.figure(figsize=(10, 8))

# Representamos las observaciones
plt.scatter(
    componentes[:, 0],
    componentes[:, 1],
    alpha=0.4,
    color='steelblue'
)

# Proyección de las variables originales
loadings = pca.components_.T

for i, variable in enumerate(Hipo.columns):
    plt.arrow(
        0, 0,
        loadings[i, 0] * 3,
        loadings[i, 1] * 3,
        color='red',
        width=0.008,
        head_width=0.08,
        length_includes_head=True
    )

    plt.text(
        loadings[i, 0] * 3.2,
        loadings[i, 1] * 3.2,
        variable,
        color='red',
        fontsize=10,
        fontweight='bold'
    )

plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('PCA: IMC, cintura y glucemia')

plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)

plt.grid(alpha=0.3)
plt.show()

#### 4.6.2. Análisis complementario: presión, edad y glucemia

In [ ]:
# Hipótesis 6: análisis complementario (presión, edad y glucemia)
# Repetimos el PCA con estas tres variables

columnas = [
    'bima04_01_a',
    'bhch04',
    'bimq05'
]

# Seleccionamos las variables
datos = df[columnas].dropna().copy()

# Renombramos las columnas
datos.rename(columns={
    'bima04_01_a': 'Presion Sistolica',
    'bhch04': 'Edad',
    'bimq05': 'Glucemia en Ayunas'
}, inplace=True)

# Estandarizamos los datos
scaler = StandardScaler()
datos_escalados = scaler.fit_transform(datos)

# Aplicamos PCA
pca = PCA(n_components=2)
componentes = pca.fit_transform(datos_escalados)

# Visualización
plt.figure(figsize=(10, 8))

plt.scatter(
    componentes[:, 0],
    componentes[:, 1],
    alpha=0.5
)

# Proyección de las variables originales
loadings = pca.components_.T

for i, variable in enumerate(datos.columns):
    plt.arrow(
        0, 0,
        loadings[i, 0] * 3,
        loadings[i, 1] * 3,
        color='red',
        width=0.008,
        head_width=0.08,
        length_includes_head=True
    )

    plt.text(
        loadings[i, 0] * 3.2,
        loadings[i, 1] * 3.2,
        variable,
        color='red',
        fontsize=10,
        fontweight='bold'
    )

plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('PCA: Presión, Edad y Glucemia')
plt.grid()
plt.show()